# Bolum 3: SQL Sorgusunu Pandas DataFrame'e Cevirme

`pd.read_sql()` ile SQL sonucunu direkt DataFrame olarak aliyoruz.

Boylece SQL'in gucu + Pandas'in esnekligi bir arada!

In [6]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    host="localhost",
    database="postgres",
    user="postgres",
    password="1234"
)

## Kategorilere gore urun sayisi

In [7]:
df_kategoriler = pd.read_sql("""
    SELECT c.category_name AS kategori, COUNT(p.product_id) AS urun_sayisi
    FROM products p
    JOIN categories c ON p.category_id = c.category_id
    GROUP BY c.category_name
    ORDER BY urun_sayisi DESC
""", conn)

df_kategoriler

/var/folders/_7/4xlcmys94rvbr77bgf94cd180000gn/T/ipykernel_64413/3754218901.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_kategoriler = pd.read_sql("""


,kategori,urun_sayisi
0,Confections,13
1,Condiments,12
2,Beverages,12
3,Seafood,12
4,Dairy Products,10
5,Grains/Cereals,7
6,Meat/Poultry,6
7,Produce,5


In [8]:
print(f"Toplam {len(df_kategoriler)} kategori var.")
print(f"En cok urune sahip kategori: {df_kategoriler.iloc[0]['kategori']}")

Toplam 8 kategori var.
En cok urune sahip kategori: Confections


## Aylik satis verisi

In [9]:
df_aylik = pd.read_sql("""
    SELECT
        DATE_TRUNC('month', o.order_date) AS ay,
        ROUND(SUM(od.unit_price * od.quantity)::numeric, 2) AS toplam_satis
    FROM orders o
    JOIN order_details od ON o.order_id = od.order_id
    GROUP BY ay
    ORDER BY ay
""", conn)

df_aylik['ay'] = pd.to_datetime(df_aylik['ay'], utc=True)

df_aylik.head(10)

/var/folders/_7/4xlcmys94rvbr77bgf94cd180000gn/T/ipykernel_64413/3705279078.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_aylik = pd.read_sql("""


,ay,toplam_satis
0,1996-06-30 21:00:00+00:00,30192.10
1,1996-07-31 21:00:00+00:00,26609.40
2,1996-08-31 21:00:00+00:00,27636.00
3,1996-09-30 21:00:00+00:00,41203.60
4,1996-10-31 22:00:00+00:00,49704.00
5,1996-11-30 22:00:00+00:00,50953.40
6,1996-12-31 22:00:00+00:00,66692.80
7,1997-01-31 22:00:00+00:00,41207.20
8,1997-02-28 22:00:00+00:00,39979.90
9,1997-03-31 21:00:00+00:00,55699.39


In [11]:
# print(f"En yuksek satis: ${df_aylik['toplam_satis'].max()}")
print(f"En yuksek satis: ${df_aylik['toplam_satis'].max():,.0f}")
# print(f"En dusuk satis: ${df_aylik['toplam_satis'].min()}")
print(f"En dusuk satis: ${df_aylik['toplam_satis'].min():,.0f}")
# print(f"Ortalama aylik satis: ${df_aylik['toplam_satis'].mean()}")
print(f"Ortalama aylik satis: ${df_aylik['toplam_satis'].mean():,.0f}")

En yuksek satis: $134,631
En dusuk satis: $19,899
Ortalama aylik satis: $58,890
